In [0]:
%sql
CREATE OR REPLACE TABLE youtube_lakehouse.gold.agg_publishing_window_heatmap AS
WITH base_video_summary AS (
    SELECT 
        video_id,
        channel_id,
        channel_title,
        video_title,
        published_at,
        DATE_FORMAT(published_at, 'EEEE') AS publish_day_name,
        DAYOFWEEK(published_at) AS publish_day_of_week,
        HOUR(published_at) AS publish_hour_utc,
        MAX(cumulative_views) AS latest_views,
        MAX(cumulative_likes) AS latest_likes,
        MAX(cumulative_comments) AS latest_comments,
        MAX(CASE WHEN days_since_published <= 2 THEN delta_views_24h ELSE 0 END) AS initial_velocity_views
    FROM youtube_lakehouse.silver.fact_video_daily_snapshots
    GROUP BY 
        video_id,
        channel_id,
        channel_title,
        video_title,
        published_at
)
SELECT 
    channel_title,
    publish_day_of_week,
    publish_day_name,
    publish_hour_utc,
    COUNT(DISTINCT video_id) AS total_videos_published,
    ROUND(AVG(initial_velocity_views), 0) AS avg_initial_velocity_views,
    ROUND(PERCENTILE_APPROX(initial_velocity_views, 0.5), 0) AS median_initial_velocity_views,
    ROUND(AVG(latest_views), 0) AS avg_views,
    ROUND(PERCENTILE_APPROX(latest_views, 0.5), 0) AS median_views,
    ROUND(
        COALESCE(
            (SUM(latest_likes) + SUM(latest_comments)) / NULLIF(SUM(latest_views), 0) * 100, 
            0.0
        ), 
        2
    ) AS engagement_rate_pct
FROM base_video_summary
GROUP BY 
    channel_title,
    publish_day_of_week,
    publish_day_name,
    publish_hour_utc;

In [0]:
%sql
CREATE OR REPLACE TABLE youtube_lakehouse.gold.agg_keyword_momentum_heatmap AS
WITH video_level_tags AS (
    SELECT 
        s.video_id,
        s.channel_title,
        s.video_title,
        s.published_at,
        t.normalized_tag,
        DATE_FORMAT(s.published_at, 'EEEE') AS publish_day_name,
        DAYOFWEEK(s.published_at) AS publish_day_of_week,
        HOUR(s.published_at) AS publish_hour_utc,
        MAX(s.cumulative_views) AS latest_views,
        MAX(s.cumulative_likes) AS latest_likes,
        MAX(s.cumulative_comments) AS latest_comments,
        MAX(CASE WHEN s.days_since_published <= 2 THEN s.delta_views_24h ELSE 0 END) AS initial_velocity_views
    FROM youtube_lakehouse.silver.fact_video_daily_snapshots s
    INNER JOIN youtube_lakehouse.silver.dim_video_tags t 
        ON s.video_id = t.video_id
    GROUP BY 
        s.video_id,
        s.channel_title,
        s.video_title,
        s.published_at,
        t.normalized_tag
)
SELECT 
    normalized_tag,
    publish_day_of_week,
    publish_day_name,
    publish_hour_utc,
    COUNT(DISTINCT video_id) AS total_videos,
    ROUND(AVG(initial_velocity_views), 0) AS avg_initial_velocity_views,
    ROUND(PERCENTILE_APPROX(initial_velocity_views, 0.5), 0) AS median_initial_velocity_views,
    ROUND(AVG(latest_views), 0) AS avg_cumulative_views,
    ROUND(PERCENTILE_APPROX(latest_views, 0.5), 0) AS median_cumulative_views,
    ROUND(
        COALESCE(
            (SUM(latest_likes) + SUM(latest_comments)) / NULLIF(SUM(latest_views), 0) * 100, 
            0.0
        ), 
        2
    ) AS tag_engagement_rate_pct
FROM video_level_tags
GROUP BY 
    normalized_tag,
    publish_day_of_week,
    publish_day_name,
    publish_hour_utc;

In [0]:
%sql
OPTIMIZE youtube_lakehouse.gold.agg_publishing_window_heatmap
ZORDER BY (publish_day_of_week, publish_hour_utc);

OPTIMIZE youtube_lakehouse.gold.agg_keyword_momentum_heatmap
ZORDER BY (normalized_tag, publish_day_of_week, publish_hour_utc);

In [0]:
%sql
OPTIMIZE youtube_lakehouse.gold.agg_publishing_window_heatmap
ZORDER BY (channel_title, publish_day_of_week, publish_hour_utc);

In [0]:
%sql
SELECT 
    publish_day_name,
    publish_hour_utc,
    total_videos_published,
    median_views,
    median_initial_velocity_views,
    engagement_rate_pct
FROM youtube_lakehouse.gold.agg_publishing_window_heatmap
ORDER BY total_videos_published DESC
LIMIT 10;

In [0]:
# Task 3.5: Automated Verification on Gold Aggregations
checks = {
    "empty_publishing_heatmap": """
        SELECT * 
        FROM youtube_lakehouse.gold.agg_publishing_window_heatmap
    """,
    "invalid_hour_range": """
        SELECT * 
        FROM youtube_lakehouse.gold.agg_publishing_window_heatmap 
        WHERE publish_hour_utc < 0 OR publish_hour_utc > 23
    """,
    "invalid_day_of_week": """
        SELECT * 
        FROM youtube_lakehouse.gold.agg_publishing_window_heatmap 
        WHERE publish_day_of_week < 1 OR publish_day_of_week > 7
    """,
    "invalid_gold_metrics": """
        SELECT * 
        FROM youtube_lakehouse.gold.agg_publishing_window_heatmap 
        WHERE total_videos_published <= 0 
           OR median_views < 0 
           OR engagement_rate_pct < 0 
           OR engagement_rate_pct > 100
    """,
    "keyword_heatmap_null_tags": """
        SELECT * 
        FROM youtube_lakehouse.gold.agg_keyword_momentum_heatmap 
        WHERE normalized_tag IS NULL OR LENGTH(normalized_tag) < 2
    """
}

# 1. Verify table populated
base_count = spark.sql(checks["empty_publishing_heatmap"]).count()
assert base_count > 0, "❌ FAILED: Gold table agg_publishing_window_heatmap is empty!"
print(f"✅ PASSED: agg_publishing_window_heatmap populated with {base_count} time slot bins.")

# 2. Run boundary and metric checks
all_passed = True
for check_name in ["invalid_hour_range", "invalid_day_of_week", "invalid_gold_metrics", "keyword_heatmap_null_tags"]:
    failed_df = spark.sql(checks[check_name])
    fail_count = failed_df.count()
    if fail_count > 0:
        print(f"❌ FAILED: {check_name} (Found {fail_count} invalid records)")
        display(failed_df.limit(5))
        all_passed = False
    else:
        print(f"✅ PASSED: {check_name}")

assert all_passed, "Gold validation checks failed."
print("\nAll Gold Layer verification audits passed successfully!")

In [0]:
%sql
CREATE OR REPLACE TABLE youtube_lakehouse.gold.dim_channels AS
SELECT 
    channel_id,
    channel_title,
    COUNT(DISTINCT video_id) AS total_videos_tracked,
    MIN(published_at) AS first_video_published_at,
    MAX(published_at) AS latest_video_published_at
FROM youtube_lakehouse.silver.fact_video_daily_snapshots
GROUP BY 
    channel_id,
    channel_title;

OPTIMIZE youtube_lakehouse.gold.dim_channels
ZORDER BY (channel_id);

In [0]:
%sql
CREATE OR REPLACE TABLE youtube_lakehouse.gold.fact_video_performance AS
WITH video_aggregates AS (
    SELECT 
        video_id,
        channel_id,
        video_title,
        published_at,
        DATE_FORMAT(published_at, 'EEEE') AS publish_day_name,
        DAYOFWEEK(published_at) AS publish_day_of_week,
        HOUR(published_at) AS publish_hour_utc,
        MAX(cumulative_views) AS views,
        MAX(cumulative_likes) AS likes,
        MAX(cumulative_comments) AS comments,
        -- Initial 48-hour velocity metric
        MAX(CASE WHEN days_since_published <= 2 THEN delta_views_24h ELSE 0 END) AS initial_velocity_views_48h
    FROM youtube_lakehouse.silver.fact_video_daily_snapshots
    GROUP BY 
        video_id,
        channel_id,
        video_title,
        published_at
)
SELECT 
    video_id,
    channel_id,
    video_title,
    published_at,
    publish_day_name,
    publish_day_of_week,
    publish_hour_utc,
    views,
    likes,
    comments,
    initial_velocity_views_48h,
    ROUND(
        COALESCE((likes + comments) / NULLIF(views, 0) * 100, 0.0), 
        2
    ) AS engagement_rate_pct
FROM video_aggregates;

OPTIMIZE youtube_lakehouse.gold.fact_video_performance
ZORDER BY (channel_id, publish_day_of_week, publish_hour_utc);